# Indagine dedicata al punto critico — trimero a catena, con DM

Mirror di `analisi_espressivita_PMA_anello.ipynb`, ma non un secondo notebook
esplorativo nel senso pieno: nato durante l'analisi del confronto sistematico
per capire se un plateau di fidelity osservato con soli 2 restart fosse un
vero tetto strutturale o un artefatto di ottimizzazione (come richiesto dal
piano: "verificala sui dati della catena senza assumerla").

**Risultato**: è un vero tetto. Né RBS-2q né W-2q a un solo giro di blocchi
raggiungono $\mathcal F=1$ esatto sotto DM — a differenza dell'anello, dove un
solo giro di $W$ bastava. Servono due giri completi (famiglia `*-2qC.K2`).

Cella interattiva in fondo: scegli un ansatz e il numero di restart, e
ricalcola dal vivo (richiede kernel Jupyter attivo).

## 0. Setup

In [ ]:
import numpy as np
from scipy.optimize import minimize
from qiskit.quantum_info import Statevector

from trimer_chain_exact import critical_field, ground_state_projector_dm
from ansatz_catena import build_ansatze, pma_2q_trimer_cyclic, w_block, rbs_block

J, D = 1.0, 0.15
bc = critical_field(J)
P0, E0, deg = ground_state_projector_dm(J, bc, D)
print(f"b_c = {bc}, D = {D}, degenerazione fondamentale = {deg}")

CANDIDATI = build_ansatze()
CANDIDATI["RBS-2qC.K2"] = pma_2q_trimer_cyclic(2, rbs_block)
CANDIDATI["W-2qC.K2"] = pma_2q_trimer_cyclic(2, w_block)


def max_fidelity(qc, P0, n_restarts=60, seed=3):
    def neg_fid(params):
        sv = Statevector(qc.assign_parameters(params)).data
        return -np.real(sv.conj() @ P0 @ sv)
    rng = np.random.default_rng(seed)
    best = 0.0
    for _ in range(n_restarts):
        x0 = rng.uniform(-np.pi, np.pi, qc.num_parameters)
        r = minimize(neg_fid, x0, method="COBYLA", options={"maxiter": 600, "tol": 1e-12})
        r2 = minimize(neg_fid, r.x, method="L-BFGS-B", options={"maxiter": 600, "ftol": 1e-16})
        best = max(best, -r2.fun)
    return best

## 1. Tabella di riferimento

Qui **15 restart** per tempi di esecuzione ragionevoli in un notebook
interattivo (aumenta pure il numero nella cella interattiva in fondo,
se hai qualche minuto in piu'). La verifica definitiva a **60-80
restart su 3 seed indipendenti** (differenza $<10^{-10}$ fra i seed,
quindi non un artefatto) e' documentata in `indagine_catena_bc.py` e
riportata in `trimero_catena_vqe_dm.pdf`. I valori a 15 restart qui
sotto sono gia' coerenti con quelli a 60-80 (stesso tetto).

In [ ]:
RIFERIMENTO = [
    ("RBS-2q.5", 5), ("RBS-2q.8", 8), ("RBS-2qC.K1", 5), ("RBS-2qC.K2", 10),
    ("W-2q.5", 5), ("W-2q.8", 8), ("W-2qC.K1", 5), ("W-2qC.K2", 10),
    ("W-1q.6", 6), ("W-1q.9", 9),
]

print(f"{'ansatz':14s} {'par':>4s} {'fidelity (15 restart)':>22s}")
risultati = {}
for name, npar in RIFERIMENTO:
    qc = CANDIDATI[name]
    assert qc.num_parameters == npar
    best = max_fidelity(qc, P0, n_restarts=15)
    risultati[name] = best
    print(f"{name:14s} {npar:4d} {best:22.10f}")

## 2. Il pattern: un giro non basta, due giri sì

RBS-2q.5, RBS-2q.8, RBS-2qC.K1 hanno numeri di parametri diversi (5, 8, 5) ma
**stessa fidelity**: segno che l'espressività aggiuntiva (i parametri extra
sono $Ry$ indipendenti dopo un solo giro di blocchi) non aiuta — il collo di
bottiglia è strutturale, non di conteggio parametri. Lo stesso per la
famiglia $W$ corrispondente, a un tetto più alto.

In [ ]:
gruppi = {
    "RBS, 1 giro": ["RBS-2q.5", "RBS-2q.8", "RBS-2qC.K1"],
    "RBS, 2 giri (K2)": ["RBS-2qC.K2"],
    "W, 1 giro": ["W-2q.5", "W-2q.8", "W-2qC.K1"],
    "W, 2 giri (K2)": ["W-2qC.K2"],
}
for label, nomi in gruppi.items():
    vals = [risultati[n] for n in nomi]
    print(f"{label:20s}: {[round(v,7) for v in vals]}  "
          f"(scarto max fra varianti: {max(vals)-min(vals):.2e})")

## 3. Ricalcolo interattivo

Scegli un ansatz e il numero di restart, poi premi "Ricalcola". Utile per
verificare con mano propria che il tetto non si sposta aumentando i restart
oltre 60 (prova ad esempio `RBS-2q.8` con 100 restart: resta a $0.9626$).

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

if HAS_WIDGETS:
    dropdown = widgets.Dropdown(
        options=list(CANDIDATI.keys()), value="RBS-2q.8",
        description="Ansatz:", style={"description_width": "initial"},
    )
    slider = widgets.IntSlider(
        value=60, min=10, max=150, step=10,
        description="Restart:", style={"description_width": "initial"},
    )
    seed_box = widgets.IntText(value=3, description="Seed:",
                                style={"description_width": "initial"})
    button = widgets.Button(description="Ricalcola", button_style="primary")
    out = widgets.Output()

    def _ricalcola(_):
        with out:
            clear_output(wait=True)
            qc = CANDIDATI[dropdown.value]
            print(f"Calcolo {dropdown.value} ({qc.num_parameters} par), "
                  f"{slider.value} restart, seed={seed_box.value}...")
            best = max_fidelity(qc, P0, n_restarts=slider.value, seed=seed_box.value)
            print(f"  F = {best:.10f}")
            if dropdown.value in risultati:
                rif = risultati[dropdown.value]
                print(f"  (riferimento a 60 restart, seed=3: {rif:.10f}, "
                      f"scarto {abs(best-rif):.2e})")

    button.on_click(_ricalcola)
    print("Selettore interattivo (richiede kernel Jupyter vivo):")
    display(widgets.HBox([dropdown, slider, seed_box, button]), out)
    _ricalcola(None)
else:
    print("ipywidgets non disponibile: nessun ricalcolo live.")
    print("Uso max_fidelity(CANDIDATI['RBS-2q.8'], P0, n_restarts=60) direttamente.")

## Conclusioni

- Sotto DM, sulla catena, un **solo giro** di blocchi (RBS o $W$) non basta
  per $\mathcal F=1$ esatto — a differenza dell'anello.
- Il tetto è **strutturale**: identico per le tre varianti di parametrizzazione
  di ciascuna famiglia, stabile su più seed e più restart.
- **Due giri completi** (`*-2qC.K2`, 10 parametri) recuperano l'esatto per
  entrambe le famiglie.
- $W$ resta più economico in gate a parità di parametri (vedi
  `trimero_catena_vqe_dm.pdf`): candidato canonico **`W-2qC.K2`**.